# 02 Feature Ideas: IEEE-CIS Fraud Detection

Это `Jupyter notebook` для следующего шага после `EDA`, `SQL` и первых `rule drafts`.

Цель:
- собрать подтвержденные anti-fraud сигналы в одном месте;
- перевести их в идеи признаков и правил;
- подготовить первый feature set для будущей baseline модели;
- не прыгать в `ML` раньше времени, а сначала понять логику признаков.


## План работы

1. Загрузить `train_transaction` и `train_identity`.
2. Зафиксировать подтвержденные сигналы из `EDA + SQL`.
3. Описать rule ideas и related feature ideas.
4. Собрать простой датафрейм с первыми фичами.
5. Проверить, как эти фичи распределяются между fraud и non-fraud.
6. Зафиксировать риски и ограничения перед baseline model.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [2]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DOCS_DIR = PROJECT_ROOT / 'docs'
TRANSACTION_PATH = DATA_DIR / 'train_transaction.csv'
IDENTITY_PATH = DATA_DIR / 'train_identity.csv'
RULES_PATH = DOCS_DIR / 'ieee_cis_fraud_detection' / 'week_1' / 'rules' / 'first_rule_drafts.md'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSACTION_PATH exists:', TRANSACTION_PATH.exists())
print('IDENTITY_PATH exists:', IDENTITY_PATH.exists())
print('RULES_PATH exists:', RULES_PATH.exists())


PROJECT_ROOT: /Users/drhtka/Downloads/Projects/Llm_ml_RAG/anti_fraud_analytics_platform
TRANSACTION_PATH exists: True
IDENTITY_PATH exists: True
RULES_PATH exists: True


In [3]:
def load_csv_if_exists(path: Path):
    if path.exists():
        print(f'Loaded: {path.name}')
        return pd.read_csv(path)
    print(f'File not found: {path}')
    return None


transactions = load_csv_if_exists(TRANSACTION_PATH)
identity = load_csv_if_exists(IDENTITY_PATH)


Loaded: train_transaction.csv
Loaded: train_identity.csv


In [4]:
if transactions is not None:
    print('transactions shape:', transactions.shape)
    display(transactions[['TransactionID', 'isFraud', 'TransactionAmt', 'ProductCD', 'card1', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'addr1']].head())

if identity is not None:
    print('identity shape:', identity.shape)
    display(identity[['TransactionID', 'DeviceType', 'DeviceInfo']].head())


transactions shape: (590540, 394)


,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card4,card6,P_emaildomain,R_emaildomain,addr1
0,2987000,0,68.5,W,13926,discover,credit,NaN,NaN,315.0
1,2987001,0,29.0,W,2755,mastercard,credit,gmail.com,NaN,325.0
2,2987002,0,59.0,W,4663,visa,debit,outlook.com,NaN,330.0
3,2987003,0,50.0,W,18132,mastercard,debit,yahoo.com,NaN,476.0
4,2987004,0,50.0,H,4497,mastercard,credit,gmail.com,NaN,420.0


identity shape: (144233, 41)


,TransactionID,DeviceType,DeviceInfo
0,2987004,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,mobile,iOS Device
2,2987010,desktop,Windows
3,2987011,desktop,NaN
4,2987016,desktop,MacOS


## Подтвержденные сигналы из EDA + SQL

На этот ноутбук мы уже приносим готовые сигналы, а не ищем их с нуля:

- `ProductCD = C` -> `11.69%` fraud rate
- `card4 = discover` -> `7.73%` fraud rate
- `card6 = credit` -> `6.68%` fraud rate
- suspicious email domains: `outlook.com`, `icloud.com`, `gmail.com`
- unusually large transaction relative to `card1` proxy

Идея этого ноутбука: превратить эти сигналы в признаки, которые позже можно подать в baseline model.


In [5]:
candidate_rules = pd.DataFrame([
    {'rule_name': 'productcd_c_flag', 'signal': 'ProductCD = C', 'evidence': 'fraud_rate_pct = 11.69%', 'mvp_usage': 'risk flag / score boost'},
    {'rule_name': 'card4_discover_flag', 'signal': 'card4 = discover', 'evidence': 'fraud_rate_pct = 7.73%', 'mvp_usage': 'risk flag / manual review boost'},
    {'rule_name': 'card6_credit_flag', 'signal': 'card6 = credit', 'evidence': 'fraud_rate_pct = 6.68%', 'mvp_usage': 'risk flag / score boost'},
    {'rule_name': 'high_risk_email_flag', 'signal': 'P/R email domain in suspicious set', 'evidence': 'outlook.com / icloud.com / gmail.com', 'mvp_usage': 'soft signal in score'},
    {'rule_name': 'amount_deviation_flag', 'signal': 'TransactionAmt > avg + 3 * std by card1', 'evidence': 'suspicious pattern SQL', 'mvp_usage': 'manual review trigger'}
])
candidate_rules


,rule_name,signal,evidence,mvp_usage
0,productcd_c_flag,ProductCD = C,fraud_rate_pct = 11.69%,risk flag / score boost
1,card4_discover_flag,card4 = discover,fraud_rate_pct = 7.73%,risk flag / manual review boost
2,card6_credit_flag,card6 = credit,fraud_rate_pct = 6.68%,risk flag / score boost
3,high_risk_email_flag,P/R email domain in suspicious set,outlook.com / icloud.com / gmail.com,soft signal in score
4,amount_deviation_flag,TransactionAmt > avg + 3 * std by card1,suspicious pattern SQL,manual review trigger


## Первый набор feature ideas

На этом этапе мы не строим идеальный production-ready feature store.

Мы делаем MVP feature set:
- простые бинарные флаги;
- понятные агрегаты;
- фичи, которые легко объяснить на интервью и в портфолио.


In [9]:
high_risk_p_domains = {'outlook.com'}
high_risk_r_domains = {'outlook.com', 'icloud.com', 'gmail.com'}

if transactions is not None:
    card1_amount_stats = (
        transactions.groupby('card1')['TransactionAmt']
        .agg(card1_amt_mean='mean', card1_amt_std='std')
    )

    feature_df = transactions[[
        'TransactionID', 'isFraud', 'TransactionAmt', 'ProductCD', 'card1', 'card4', 'card6',
        'P_emaildomain', 'R_emaildomain', 'addr1', 'TransactionDT'
    ]].copy()

    feature_df = feature_df.join(card1_amount_stats, on='card1')

    feature_df['feat_productcd_c_flag'] = (feature_df['ProductCD'] == 'C').astype(int)
    feature_df['feat_card4_discover_flag'] = (feature_df['card4'] == 'discover').astype(int)
    feature_df['feat_card6_credit_flag'] = (feature_df['card6'] == 'credit').astype(int)
    feature_df['feat_high_risk_p_email_flag'] = feature_df['P_emaildomain'].isin(high_risk_p_domains).astype(int)
    feature_df['feat_high_risk_r_email_flag'] = feature_df['R_emaildomain'].isin(high_risk_r_domains).astype(int)
    feature_df['feat_missing_r_email_flag'] = feature_df['R_emaildomain'].isna().astype(int)

    threshold = feature_df['card1_amt_mean'] + 3 * feature_df['card1_amt_std'].fillna(0)
    feature_df['feat_amount_gt_card1_avg_plus_3std'] = (feature_df['TransactionAmt'] > threshold).astype(int)
    feature_df['feat_amount_log1p'] = np.log1p(feature_df['TransactionAmt'])

    display(feature_df.head())


,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card4,card6,P_emaildomain,R_emaildomain,addr1,TransactionDT,card1_amt_mean,card1_amt_std,feat_productcd_c_flag,feat_card4_discover_flag,feat_card6_credit_flag,feat_high_risk_p_email_flag,feat_high_risk_r_email_flag,feat_missing_r_email_flag,feat_amount_gt_card1_avg_plus_3std,feat_amount_log1p
0,2987000,0,68.5,W,13926,discover,credit,NaN,NaN,315.0,86400,351.931163,371.141254,0,1,1,0,0,1,0,4.241327
1,2987001,0,29.0,W,2755,mastercard,credit,gmail.com,NaN,325.0,86401,234.292753,460.356975,0,0,1,0,0,1,0,3.401197
2,2987002,0,59.0,W,4663,visa,debit,outlook.com,NaN,330.0,86469,97.015542,100.128858,0,0,0,1,0,1,0,4.094345
3,2987003,0,50.0,W,18132,mastercard,debit,yahoo.com,NaN,476.0,86499,123.416340,192.717425,0,0,0,0,0,1,0,3.931826
4,2987004,0,50.0,H,4497,mastercard,credit,gmail.com,NaN,420.0,86506,96.972222,56.629451,0,0,1,0,0,1,0,3.931826


## Быстрая проверка фичей

Здесь мы пока не обучаем модель.

Мы просто смотрим, отличаются ли fraud и non-fraud по нашим простым feature flags.


In [ ]:
feature_cols = [
    'feat_productcd_c_flag',
    'feat_card4_discover_flag',
    'feat_card6_credit_flag',
    'feat_high_risk_p_email_flag',
    'feat_high_risk_r_email_flag',
    'feat_missing_r_email_flag',
    'feat_amount_gt_card1_avg_plus_3std',
    'feat_amount_log1p',
]

if transactions is not None:
    feature_summary = feature_df.groupby('isFraud')[feature_cols].mean().T
    feature_summary.columns = ['non_fraud_mean', 'fraud_mean']
    feature_summary['fraud_minus_non_fraud'] = feature_summary['fraud_mean'] - feature_summary['non_fraud_mean']
    display(feature_summary.sort_values('fraud_minus_non_fraud', ascending=False))


,non_fraud_mean,fraud_mean,fraud_minus_non_fraud
feat_productcd_c_flag,0.106183,0.387553,0.281370
feat_high_risk_r_email_flag,0.094138,0.358370,0.264232
feat_card6_credit_flag,0.243975,0.481537,0.237562
feat_high_risk_p_email_flag,0.008096,0.023327,0.015230
feat_card4_discover_flag,0.010769,0.024875,0.014106
feat_amount_gt_card1_avg_plus_3std,0.017856,0.023569,0.005712
feat_amount_log1p,4.383279,4.374155,-0.009124
feat_missing_r_email_flag,0.778787,0.456662,-0.322126


## Риски и ограничения

Важно помнить:

- `card1` это только customer-like proxy, а не настоящий customer id;
- `TransactionDT` это relative time, а не нормальный timestamp;
- email domain и card-сегменты могут давать `false positives`;
- rule-based сигналы полезны, но сами по себе не заменяют модель.


## Что пойдет дальше

После этого ноутбука нужно получить:

- 5-10 осмысленных признаков-кандидатов;
- понимание, какие признаки оставить в baseline model;
- список правил, которые можно использовать в MVP anti-fraud scoring.

Следующий шаг после этого ноутбука: `03_baseline_model.ipynb`.


## Первые выводы по feature ideas

По простому сравнению fraud vs non-fraud самыми сильными кандидатами в baseline feature set выглядят:

- `feat_productcd_c_flag`
- `feat_high_risk_r_email_flag`
- `feat_card6_credit_flag`

Дополнительно можно оставить:

- `feat_high_risk_p_email_flag`
- `feat_card4_discover_flag`

Более слабые или спорные признаки:

- `feat_amount_gt_card1_avg_plus_3std`
- `feat_amount_log1p`

Отдельно важно:
- `feat_missing_r_email_flag` имеет отрицательный сдвиг, но это не делает его бесполезным;
- такой признак тоже может быть полезен модели как anti-signal.

In [11]:
initial_feature_cols = [
    'feat_productcd_c_flag',
    'feat_high_risk_r_email_flag',
    'feat_card6_credit_flag',
    'feat_high_risk_p_email_flag',
    'feat_card4_discover_flag',
    'feat_missing_r_email_flag',
    'feat_amount_gt_card1_avg_plus_3std',
    'feat_amount_log1p',
]

display(feature_df[initial_feature_cols + ['isFraud']].head())

,feat_productcd_c_flag,feat_high_risk_r_email_flag,feat_card6_credit_flag,feat_high_risk_p_email_flag,feat_card4_discover_flag,feat_missing_r_email_flag,feat_amount_gt_card1_avg_plus_3std,feat_amount_log1p,isFraud
0,0,0,1,0,1,1,0,4.241327,0
1,0,0,1,0,0,1,0,3.401197,0
2,0,0,0,1,0,1,0,4.094345,0
3,0,0,0,0,0,1,0,3.931826,0
4,0,0,1,0,0,1,0,3.931826,0
